## Data Collection and Deduplication of Funding Data
Brief summary:
1. Import previous year's data.
2. Import global airtable data. Filter by Europe only and date added after 1 Jan 20XX.
3. Collect Dimensions data and deduplicate across searches.
4. Import Bruna contractor data.
5. Run Dimensions and Bruna data through scope check and AP pillar labelling.
6. Combination & deduplication: global grants tracker into last year's data
7. Combination & deduplication: Dimensions results into last year's data
8. Combination & deduplication: Bruna contractor data into last year's data

In [1]:
## Imports and config ##

import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
import pycountry

load_dotenv()

OUTPUT_DIR = Path(".")
raw_data_dir = Path("1_deduplication/raw_data")

### 0. Import Finished 2026 Data
This is the data used in the 2026 reports, including data up to the end of 2025. 
The aim is to finish with this dataset, starting from the 2025 data.

In [2]:
# Import data and view top 10 rows just to check
funding_2026_data = pd.read_excel(raw_data_dir / "Funding2026_inscope.xlsx")

funding_2026_data.head(10)

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,3173601.480,3173601.480,3173601.48,3173601.48,NaN,NaN,NaN,NaN,NaN,NaN
6,Germany Earmarks €38M Investment for Alt-Prote...,NaN,NaN,airtable,38000000,38000000,EUR,41115050.0,41115050.0,38000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Solar Foods receives a €34 million grant to ra...,NaN,NaN,airtable,34000000,34000000,EUR,37298680.0,37298680.0,34000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,NaN,airtable,27600000,27600000,NOK,2604048.0,2604048.0,2401200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SYLPLANT,"Arbiom, an agricultural-biotech company develo...",NaN,airtable,23000000,14000000,EUR,25829000.0,15722000.0,23000000.0,...,3500000.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# OPTIONAL: View data types in each column to check for any issues
print(funding_2026_data.dtypes.to_string())
print()

Title                                                str
Abstract                                             str
Original title                                       str
Database                                             str
Total amount                                      object
Gov contribution                                  object
Currency                                             str
Total amount (USD)                               float64
Gov contribution (USD)                           float64
Total amount (EUR)                               float64
Gov & NP contribution (EUR)                      float64
Funding decision                                     str
copy to external database                            str
URL for announcement                                 str
Identification code                               object
Unnamed: 15                                      float64
Unnamed: 16                                      float64
Notes (external)               

In [4]:
# Full datetime columns
datetime_cols = [
    'Date request submitted', 'Date award announced',
    'Project start date', 'Date added', 'Last modified',
]
for col in datetime_cols:
    if col in funding_2026_data.columns:
        funding_2026_data[col] = pd.to_datetime(funding_2026_data[col], errors='coerce').astype('datetime64[us]')

# Year-only columns → nullable integer
year_cols = ['Year request submitted', 'Year project started', 'End date']
for col in year_cols:
    if col in funding_2026_data.columns:
        funding_2026_data[col] = pd.to_numeric(funding_2026_data[col], errors='coerce').astype('Int64')

# Duration of award (months): keep integers, null out the stray date value
def _clean_months(x):
    if pd.isna(x) or isinstance(x, pd.Timestamp):
        return None
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return None

funding_2026_data['Duration of award (months)'] = (
    funding_2026_data['Duration of award (months)'].apply(_clean_months).astype('Int64')
)

# duration (years): keep integers 0–20 only; null out dates and bad calculations
# (negative values come from missing end dates; >20 are formula errors)
def _clean_years(x):
    if pd.isna(x) or isinstance(x, pd.Timestamp):
        return None
    try:
        val = int(float(x))
        return val if 0 <= val <= 20 else None
    except (ValueError, TypeError):
        return None

funding_2026_data['duration (years)'] = (
    funding_2026_data['duration (years)'].apply(_clean_years).astype('Int64')
)

funding_2026_data.head()

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
con = duckdb.connect("funding.db")
con.execute("CREATE OR REPLACE TABLE funding_inscope AS SELECT * FROM funding_2026_data")
con.close()

print(f"Loaded {len(funding_2026_data)} rows into funding.db → table: funding_inscope")

Loaded 1678 rows into funding.db → table: funding_inscope


### 1. Import Previous Year's Data

In [6]:
# Import data and view top 10 rows just to check
last_year_data = pd.read_excel(raw_data_dir / "Funding2025_inscope.xlsx")

last_year_data.head(10)


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,2023-04-13,2023-12-04,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021.0,2030-06-10 00:00:00
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,NaN,NaN,2023-02-27,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022.0,2027-10-11 00:00:00
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,2024-03-26,2025-05-02,NaN,NaN,NaN,Tier 4 (No GFI involvement),2023.0,2025-01-06 00:00:00
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,2023-02-28,2023-04-20,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021.0,2025-01-01 00:00:00
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,NaN,NaN,2023-04-04,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021.0,2027-07-06 00:00:00
5,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NaN,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.0,...,NaN,NaN,2023-09-08,2025-05-02,NaN,NaN,checked,Tier 2 (Significant GFI involvement),2024.0,2029-11-08 00:00:00
6,Germany Earmarks €38M Investment for Alt-Prote...,NaN,NaN,airtable,38000000,38000000,EUR,41115050.0,41115050.0,38000000.0,...,NaN,NaN,2023-12-04,2023-12-04,NaN,NaN,NaN,Tier 2 (Significant GFI involvement),2024.0,2025-01-01 00:00:00
7,Solar Foods receives a €34 million grant to ra...,NaN,NaN,airtable,34000000,34000000,EUR,37298680.0,37298680.0,34000000.0,...,NaN,NaN,2023-04-18,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022.0,22/12/2025
8,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,NaN,airtable,27600000,27600000,NOK,2604048.0,2604048.0,2401200.0,...,NaN,NaN,2023-04-21,2023-04-21,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021.0,2025-01-04 00:00:00
9,SYLPLANT,"Arbiom, an agricultural-biotech company develo...",NaN,airtable,23000000,14000000,EUR,25829000.0,15722000.0,23000000.0,...,NaN,NaN,2023-07-13,2023-07-16,NaN,NaN,NaN,Tier 4 (No GFI involvement),2023.0,2026-01-06 00:00:00


In [7]:
# OPTIONAL: View data types in each column to check for any issues
print(last_year_data.dtypes.to_string())
print()

Title                                                str
Abstract                                             str
Original title                                       str
Database                                             str
Total amount                                      object
Gov contribution                                  object
Currency                                             str
Total amount (USD)                               float64
Gov contribution (USD)                           float64
Total amount (EUR)                               float64
Gov & NP contribution (EUR)                      float64
Funding decision                                     str
copy to external database                            str
URL for announcement                              object
Identification code                               object
Unnamed: 15                                      float64
Unnamed: 16                                      float64
Notes (external)               

In [8]:
# Full datetime columns
datetime_cols = [
    'Date request submitted', 'Date award announced',
    'Project start date', 'Date added', 'Last modified',
]
for col in datetime_cols:
    if col in last_year_data.columns:
        last_year_data[col] = pd.to_datetime(last_year_data[col], errors='coerce').astype('datetime64[us]')

# Year-only columns → nullable integer
year_cols = ['Year request submitted', 'Year project started', 'End date']
for col in year_cols:
    if col in last_year_data.columns:
        last_year_data[col] = pd.to_numeric(last_year_data[col], errors='coerce').astype('Int64')

# Duration of award (months): keep integers, null out the stray date value
def _clean_months(x):
    if pd.isna(x) or isinstance(x, pd.Timestamp):
        return None
    try:
        return int(float(x))
    except (ValueError, TypeError):
        return None

last_year_data['Duration of award (months)'] = (
    last_year_data['Duration of award (months)'].apply(_clean_months).astype('Int64')
)

# Note: there is no duration (years) column in the 2025 data, so we don't need to clean that here.

last_year_data.head()

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,2023-04-13,2023-12-04,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,NaN,NaN,2023-02-27,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022,<NA>
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,2024-03-26,2025-05-02,NaN,NaN,NaN,Tier 4 (No GFI involvement),2023,<NA>
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,2023-02-28,2023-04-20,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,NaN,NaN,2023-04-04,2023-04-19,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>


### 2. Import 2025 Grants Tracker Data

In [9]:
grants_tracker_data = pd.read_excel(raw_data_dir / "GrantsTracker_2026-06-30.xlsx")

grants_tracker_data['EXT_Date added'] = pd.to_datetime(grants_tracker_data['EXT_Date added'], errors='coerce')

europe_filter = (
    (grants_tracker_data['EXT_Funder region'] == 'Europe') |
    (grants_tracker_data['EXT_PI organization region'] == 'Europe')
)
date_filter = (
    (grants_tracker_data['EXT_Date added'] >= '2025-01-01') &
    (grants_tracker_data['EXT_Date added'] <= '2025-12-31')
)

grants_tracker_data = grants_tracker_data[europe_filter & date_filter].reset_index(drop=True)

print(f"{len(grants_tracker_data)} grants included after filtering")

# Output = 550 grants included. Stella's methodology said 548 grants so that's close enough.

# Rename Gov't → Gov throughout the pipeline to avoid apostrophe quoting issues
grants_tracker_data = grants_tracker_data.rename(columns={
    "INT_Gov't contribution (actual currency)": 'INT_Gov contribution (actual currency)',
    "EXT_Gov't contribution (USD)":             'EXT_Gov contribution (USD)',
})

grants_tracker_data.head()

550 grants included after filtering


,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_2035 expenditures (#),INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags
0,Systems Biology of Hydrogen Oxidising Bacteria...,0,0,GBP,0,0.0,Awarded,https://gtr.ukri.org/projects?ref=studentship-...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,The world's population is predicted to reach 1...,grant.9452322,NaN,PF,NaN
1,Food processing residues to climate smart food...,800000,800000,SEK,76376,76376.0,NaN,https://www.vr.se/swecris.html#/project/2022-0...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.4456273,NaN,BF,NaN
2,MET2FOOD,108738,108738,GBP,139436,139436.0,NaN,https://gtr.ukri.org/projects?ref=91600,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.9555967,NaN,BF,NaN
3,Understanding the functional properties of mic...,0,0,GBP,0,0.0,Awarded,https://gtr.ukri.org/projects?ref=studentship-...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,Abstract\nThere is growing demand for sustaina...,grant.13022069,NaN,BF,NaN
4,Machine-learning generated nucleases for accel...,50000,50000,GBP,61439,61439.0,NaN,https://gtr.ukri.org/projects?ref=10072768,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.13883411,NaN,PF,NaN


In [10]:
# Clean EXT_PI organization country: null out anything that isn't a recognised country name.
# Uses pycountry (ISO 3166-1) as the authoritative list, plus common aliases not covered
# by the official standard. Personal names and anything else that doesn't match are set to None.
# Semicolon- or comma-separated country lists are kept if the first entry is a valid country.

import re

valid_countries = set()
for c in pycountry.countries:
    valid_countries.add(c.name.strip().lower())
    if hasattr(c, 'common_name'):
        valid_countries.add(c.common_name.strip().lower())

# Common names not present in pycountry's official or common_name fields
valid_countries.update({'czech republic', 'russia', 'turkey', 'uk'})

def _is_valid_country_cell(val):
    if pd.isna(val) or str(val).strip() == '':
        return False
    first_part = re.split(r'[;,]', str(val))[0].strip().lower()  # handle both ; and , as separators
    return first_part in valid_countries

before_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
removed_vals = (
    grants_tracker_data['EXT_PI organization country']
    .dropna()
    .loc[lambda s: ~s.apply(_is_valid_country_cell)]
    .unique()
)

grants_tracker_data['EXT_PI organization country'] = grants_tracker_data['EXT_PI organization country'].apply(
    lambda val: val if _is_valid_country_cell(val) else None
)

after_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
print(f"Nulled out {before_count - after_count} non-country values ({before_count} → {after_count} filled)")
print(f"Values removed: {sorted(removed_vals)}")

Nulled out 29 non-country values (467 → 438 filled)
Values removed: ['Alain Baranger', 'Andre Rastica', 'Andrew Clayton', 'Andrew Stacey', 'Aurélien Ducrey, Aurélien Ducrey', 'Christer Heimtoft', 'D.K. Karefyllakis', 'Diana Maria Condeço Marques', 'Diego Moretti, Laila Hammer, Hilaj Nikolin, Pornpimol Scheuchzer', 'Ecevit Yilmaz', 'Eric Öste', 'Ernst Langthaler', 'Fabian Pfrengle', 'Fengzheng Gao, Fengzheng Gao', 'Filipa Soares', 'Gunnar Backman', 'Harriet Gregory', 'Karen Fairlie-Clarke', 'Karima Karagussova', 'Kevin STEPHENS', 'Ky Son Chu', 'Leif Horsfelt Skibsted', 'Sarah Gaunt', 'Steve Skill', 'Stig A. Borgvang', 'Thomas Brunner, Bao Duong Pham, Mathilde Delley, Barbara Franco Lucas, Franziska Götze, Isabel Häberlil, Reto Huwiler, Evelyn Markoni', 'Veronika Temml', 'Véronique Cheynier']


### 3. Dimensions Data Curation

#### a. Import and deduplicate across searches

In [11]:
### Compile individual Dimensions searches and deduplicate ###

dimensions_files = [
    "Dimensions-Grant-2026-03-17_10-54-55_PB.xlsx",
    "Dimensions-Grant-2026-03-17_10-55-57_Ferm.xlsx",
    "Dimensions-Grant-2026-03-17_10-56-52_CM.xlsx",
    "Dimensions-Grant-2026-03-17_11-55-08_all.xlsx",
]

dfs = [pd.read_excel(raw_data_dir / f, skiprows=1) for f in dimensions_files] #skiprows=1 to skip the first row of each file, which contains search information from Dimensions
dimensions_data = pd.concat(dfs, ignore_index=True)

before = len(dimensions_data)
dimensions_data = dimensions_data.drop_duplicates(subset="Grant ID")
after = len(dimensions_data)

print(f"Removed {before - after} duplicates ({before} → {after} rows)")

### 57 grants deduplicated from within combined Dimensions data - same as Stella ###
dimensions_data.head()

Removed 57 duplicates (277 → 220 rows)


,Rank,Grant ID,Grant Number(s),Title,Title translated,Abstract,Abstract translated,Keywords,Funding amount,Currency,...,Fields of Research (ANZSRC 2020),RCDC Categories,HRCS HC Categories,HRCS RAC Categories,Health Research Areas,Broad Research Areas,Cancer Types,CSO Categories,Units of Assessment,Sustainable Development Goals
0,1471,grant.14704264,25-26-00163,"Vegetable drink based on oats Avena sativa, en...","Vegetable drink based on oats Avena sativa, en...",Macro- and microelements are necessary for mos...,Macro- and microelements are necessary for mos...,iron; vitamin B9; essential amino acids; tripl...,0,NaN,...,32 Biomedical and Clinical Sciences; 3210 Nutr...,Complementary and Integrative Health; Dietary ...,NaN,3.3 Nutrition and chemoprevention,NaN,Public Health,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger
1,1408,grant.15048923,370795,Designed fermentation for off-flavor eliminati...,Designed fermentation for off-flavor eliminati...,Plant-based foods like those made from faba be...,Plant-based foods like those made from faba be...,Food Sciences,727790,EUR,...,32 Biomedical and Clinical Sciences; 3205 Medi...,Dietary Supplements; Machine Learning and Arti...,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",NaN
2,1219,grant.15053997,10148309,PROHEMPOTIC: Advancing the Development of Hemp...,PROHEMPOTIC: Advancing the Development of Hemp...,Our project is driving innovation in sustainab...,Our project is driving innovation in sustainab...,NaN,100000,GBP,...,"30 Agricultural, Veterinary and Food Sciences;...",Nutrition,Cancer; Metabolic and endocrine; Oral and gast...,3.3 Nutrition and chemoprevention,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",12 Responsible Consumption and Production; 2 Z...
3,862,grant.15074917,126.514 IP-LS,Valorization of a starch-rich side-stream from...,Valorization of a starch-rich side-stream from...,This research project aims to develop customiz...,This research project aims to develop customiz...,NaN,458850,CHF,...,"30 Agricultural, Veterinary and Food Sciences;...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",NaN
4,791,grant.14917108,101182843,Scientific Exchange to assess QUality and Risk...,Scientific Exchange to assess QUality and Risk...,The goal of SEQUR FOOD is to build an internat...,The goal of SEQUR FOOD is to build an internat...,novel foods; bioactive food packaging; shelf l...,1656000,EUR,...,"30 Agricultural, Veterinary and Food Sciences;...",Nutrition,Metabolic and endocrine,3.3 Nutrition and chemoprevention,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",2 Zero Hunger


The next section:
1. Maps Dimensions data to last year's database via Dimensions ID / Identification code.
2. For any matches, checks for any gaps in last year's database and adds data from Dimensions if it has the data (nothing is overwritten, it is assumed last year's database is complete, so this is only filling gaps). The exception for overwriting is funding columns Funding total and Funding total (USD), which are overwritten only when the data contains a 0 value.
3. Removes any matched data from the Dimensions data, so that only new data remain.
4. Prints a summary of the matched ID's, cells edited, and rows removed (matched ID's and rows removed should be equal).
5. Saves an Excel file with only matched ID's where Dimensions data was copied into last year's data, with the specific changes highlighted in green.

Note: There is special logic for researcher names. Dimensions only has one data column called 'Researchers'. As such, the first name is assumed to be the PI and is assigned to Project lead (PI) in last year's data, while the remaining names are assigned to Collaborator names.


In [12]:
def _is_empty(val):
    if pd.isna(val):  # check for NaN / None / pd.NA
        return True
    return str(val).strip() == ''  # also treat blank strings as empty

def _is_zero(val):
    """Return True if val is numerically zero — used to overwrite placeholder 0s in funding columns."""
    if _is_empty(val):
        return False
    try:
        return float(val) == 0
    except (ValueError, TypeError):
        return False

# Work on a copy so last_year_data stays unchanged for inspection/comparison
last_year_data_edited = last_year_data.copy()

# Mapping: dimensions_data column → last_year_data column (gap-fill only, no overwrites)
col_map = {
    'Title translated':                              'Title',
    'Title':                                         'Original title',
    'Abstract translated':                           'Abstract',
    'Funding amount':                                'Total amount',
    'Currency':                                      'Currency',
    'Funding amount in USD':                         'Total amount (USD)',
    'Start date':                                    'Project start date',
    'Start Year':                                    'Year project started',
    'End Year':                                      'End date',
    'Research Organization - original':              'Collaborator institutions',
    'Research Organization - standardized':          'PI organisation',
    'State of standardized research organization':   'PI organisation state',
    'Country of standardized research organization': 'PI organisation country',
    'Funder':                                        'Funder name',
    'Funder Country':                                'Funder Country',
    'Source Linkout':                                'URL for announcement',
}

# Funding columns where a 0 value is treated as missing and can be overwritten
funding_cols_lyd = {'Total amount', 'Total amount (USD)'}

# Build a lookup dict so we can find last_year_data rows by Identification code quickly
lyd_id_map = {}
for idx, val in last_year_data['Identification code'].items():  # loop over every row in last_year_data
    if not _is_empty(val):  # skip rows with no Identification code
        lyd_id_map.setdefault(str(val).strip(), []).append(idx)

matched_ids     = set()
changed_indices = set()  # track which rows were modified, for inspection below
cells_filled    = 0

for _, dim_row in dimensions_data.iterrows():  # loop over every grant in dimensions_data
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in lyd_id_map:  # skip if this Grant ID doesn't exist in last_year_data
        continue

    matched_ids.add(grant_id)

    for lyd_idx in lyd_id_map[grant_id]:  # loop over matching rows in last_year_data (usually just one)

        # --- Simple column gap-fill ---
        for dim_col, lyd_col in col_map.items():  # loop over each column pair in the mapping
            if dim_col not in dimensions_data.columns or lyd_col not in last_year_data.columns:  # skip if either column doesn't exist in its dataset
                continue
            dim_val = dim_row[dim_col]
            if _is_empty(dim_val):  # skip if the Dimensions value is empty — nothing to fill with
                continue
            target_val = last_year_data_edited.at[lyd_idx, lyd_col]
            overwrite_zero = lyd_col in funding_cols_lyd and _is_zero(target_val)  # for funding cols, treat 0 as fillable
            if _is_empty(target_val) or overwrite_zero:  # fill if empty, or if it's a 0 in a funding column
                last_year_data_edited.at[lyd_idx, lyd_col] = dim_val
                changed_indices.add(lyd_idx)
                cells_filled += 1

        # --- Researchers → Project lead (PI) + Collaborator names ---
        researchers_val = dim_row.get('Researchers')
        if not _is_empty(researchers_val):  # only proceed if Dimensions has researcher data
            names = [n.strip() for n in str(researchers_val).split(';') if n.strip()]  # split semicolon-separated names into a list
            if names:  # guard against an empty list after splitting
                pi_empty     = _is_empty(last_year_data_edited.at[lyd_idx, 'Project lead (PI)'])
                collab_empty = _is_empty(last_year_data_edited.at[lyd_idx, 'Collaborator names'])

                if pi_empty:  # PI field is blank — fill first researcher into PI, rest into Collaborators
                    last_year_data_edited.at[lyd_idx, 'Project lead (PI)'] = names[0]
                    changed_indices.add(lyd_idx)
                    cells_filled += 1
                    if collab_empty and len(names) > 1:  # only fill Collaborators if it's also blank and there are additional names
                        last_year_data_edited.at[lyd_idx, 'Collaborator names'] = '; '.join(names[1:])
                        cells_filled += 1
                elif collab_empty and len(names) > 1:  # PI is already filled but Collaborators is blank — add all names except the first (assumed to be the PI)
                    last_year_data_edited.at[lyd_idx, 'Collaborator names'] = '; '.join(names[1:])
                    changed_indices.add(lyd_idx)
                    cells_filled += 1

# Remove matched rows from dimensions_data — they already exist in last_year_data
before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

### 25 grants matched and removed from Dimensions data - same as Stella ###

print(f"Matched {len(matched_ids)} grants with last_year_data")
print(f"Filled {cells_filled} missing values across {len(changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 25 grants with last_year_data
Filled 80 missing values across 25 rows
Removed 25 rows from dimensions_data (195 remaining)


In [13]:
# Inspect changes: all columns, only rows that were modified, changed cells highlighted green.

lyd_changes_view = last_year_data_edited.loc[sorted(changed_indices)]

def _highlight_filled(data):
    """Green background on cells that were empty in the original but filled in the edited version."""
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        for col in data.columns:
            if _is_empty(last_year_data.at[idx, col]) and not _is_empty(data.at[idx, col]):
                styles.at[idx, col] = 'background-color: #c6efce; color: #276221'
    return styles

print(f"{len(changed_indices)} rows modified")
styled = lyd_changes_view.style.apply(_highlight_filled, axis=None)
styled.to_excel("1_deduplication/data_changes/dimensions_changes_last_year_data.xlsx", index=True)
print("Saved → 1_deduplication/data_changes/dimensions_changes_last_year_data.xlsx")
lyd_changes_view

25 rows modified
Saved → 1_deduplication/data_changes/dimensions_changes_last_year_data.xlsx


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
250,Inktelligent Foods – enabling cultivated seafo...,Cultivated meat and seafood are a sustainable ...,Inktelligent Foods – enabling cultivated seafo...,airtable,300000,300000,EUR,327925.0,327925.0,300000.000,...,NaN,NaN,2024-07-18,2025-01-10,NaN,NaN,NaN,NaN,2024,<NA>
587,Strengthening the Capacity of Excellence Hubs ...,Three key global drivers underpin the increase...,Strengthening the Capacity of Excellence Hubs ...,airtable,5946369,5946369,EUR,6436886.0,NaN,5946369.000,...,NaN,NaN,2024-12-23,2025-03-31,NaN,NaN,NaN,NaN,2025,2028
608,Scientific Exchange to assess QUality and Risk...,The goal of SEQUR FOOD is to build an internat...,Scientific Exchange to assess QUality and Risk...,airtable,1656000,1656000,EUR,1792594.0,NaN,1656000.000,...,NaN,NaN,2024-12-23,2025-03-31,NaN,NaN,NaN,NaN,2025,2028
629,PLant based EmulsifierS with improved technolo...,There is an urgent need for a new generation\n...,PLant based EmulsifierS with improved technolo...,airtable,181153,181153,EUR,196089.0,NaN,181153.000,...,NaN,NaN,2024-11-11,2025-03-31,NaN,NaN,NaN,NaN,2025,2027
658,SEACUTERIE: Crafting cultivated seafood from o...,The global population is expected to exceed ni...,SEACUTERIE: Crafting cultivated seafood from o...,airtable,50000,50000,EUR,56849.0,NaN,50000.000,...,NaN,NaN,2025-03-17,2025-04-14,NaN,NaN,NaN,NaN,2025,<NA>
660,Production of superior meat analogues by bridg...,PLANTOMYC is a transformative initiative aimin...,Production of superior meat analogues by bridg...,airtable,4618368,4618368,EUR,4999337.0,NaN,4618368.000,...,NaN,NaN,2025-03-17,2025-03-31,NaN,NaN,NaN,NaN,2025,2028
669,Impact of Fermentation on the Technofunctional...,The production of high-quality plant protein a...,Impact of Fermentation on the Technofunctional...,airtable,249618,249618,EUR,270199.0,NaN,249618.000,...,NaN,NaN,2025-03-20,2025-03-31,NaN,NaN,NaN,NaN,2025,2027
670,Plant-based Proteins for Health and Wellbeing ...,Efforts to limit the environmental impact from...,Plant-based Proteins for Health and Wellbeing ...,airtable,40000000,40000000,SEK,3790407.0,NaN,3640000.000,...,NaN,NaN,2025-03-20,2025-03-31,NaN,NaN,NaN,NaN,2025,2028
671,Harnessing the immense potential of precision ...,Melt&Marble (M&M) is a Sweden-based company th...,Harnessing the immense potential of precision ...,airtable,2485840,2485840,EUR,2690896.0,NaN,2485840.000,...,NaN,NaN,2025-03-20,2025-03-31,NaN,NaN,NaN,NaN,2025,2026
672,Novel precision fermentation process to produc...,Eggs are critically important in the global fo...,Novel precision fermentation process to produc...,airtable,2499999,2499999,EUR,2706222.0,NaN,2499999.000,...,NaN,NaN,2025-03-20,2025-03-31,NaN,NaN,NaN,NaN,2025,2026


The next section:
1. Maps Dimensions data to the newly downloaded grant tracker data via Dimensions ID / Identification code.
2. For any matches, checks for any gaps in last year's database and adds data from Dimensions if it has the data (nothing is overwritten, it is assumed last year's database is complete, so this is only filling gaps).
3. Removes any matched data from the Dimensions data, so that only new data remain.
4. Prints a summary of the matched ID's, cells edited, and rows removed (matched ID's and rows removed should be equal).
5. Saves an Excel file with only matched ID's where Dimensions data was copied into last year's data, with the specific changes highlighted in green.

Note: There is special logic for researcher names. Dimensions only has one data column called 'Researchers'. As such, the first name is assumed to be the PI and is assigned to Project lead (PI) in last year's data, while the remaining names are assigned to Collaborator names.

In [14]:
# Work on a copy so grants_tracker_data stays unchanged for inspection/comparison
grants_tracker_data_edited = grants_tracker_data.copy()


# Mapping: dimensions_data column → grants_tracker_data column (gap-fill only, no overwrites)
gt_col_map = {
    'Title translated':                              'EXT_Title',
    'Abstract translated':                           'EXT_Abstract',
    'Funding amount':                                'INT_Total amount (actual currency)',
    'Currency':                                      'INT_Currency type',
    'Funding amount in USD':                         'EXT_Total amount (USD)',
    'Start date':                                    'EXT_Project start date (estimated)',
    'End Year':                                      'INT_End date',
    'Research Organization - original':              'EXT_Collaborator organizations',
    'Research Organization - standardized':          'EXT_PI organization',
    'State of standardized research organization':   'EXT_PI organisation state',
    'Country of standardized research organization': 'EXT_PI organization country',
    'Funder':                                        'EXT_Funder name',
    'Funder Country':                                'EXT_Funder country',
    'Source Linkout':                                'EXT_URL for announcement',
}

# Funding columns where a 0 value is treated as missing and can be overwritten
funding_cols_gt = {'INT_Total amount (actual currency)', 'EXT_Total amount (USD)'}

# Build a lookup dict so we can find grants_tracker_data rows by Dimensions.ai grant ID quickly
gt_id_map = {}
for idx, val in grants_tracker_data['Dimensions.ai grant ID'].items():  # loop over every row in grants_tracker_data
    if not _is_empty(val):  # skip rows with no Dimensions ID
        gt_id_map.setdefault(str(val).strip(), []).append(idx)

gt_matched_ids     = set()
gt_changed_indices = set()  # track which rows were modified, for inspection below
gt_cells_filled    = 0

for _, dim_row in dimensions_data.iterrows():  # loop over every grant in dimensions_data
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in gt_id_map:  # skip if this Grant ID doesn't exist in grants_tracker_data
        continue

    gt_matched_ids.add(grant_id)

    for gt_idx in gt_id_map[grant_id]:  # loop over matching rows in grants_tracker_data (usually just one)

        # --- Simple column gap-fill ---
        for dim_col, gt_col in gt_col_map.items():  # loop over each column pair in the mapping
            if dim_col not in dimensions_data.columns or gt_col not in grants_tracker_data_edited.columns:  # skip if either column doesn't exist in its dataset
                continue
            dim_val = dim_row[dim_col]
            if _is_empty(dim_val):  # skip if the Dimensions value is empty — nothing to fill with
                continue
            target_val = grants_tracker_data_edited.at[gt_idx, gt_col]
            overwrite_zero = gt_col in funding_cols_gt and _is_zero(target_val)  # for funding cols, treat 0 as fillable
            if _is_empty(target_val) or overwrite_zero:  # fill if empty, or if it's a 0 in a funding column
                grants_tracker_data_edited.at[gt_idx, gt_col] = dim_val
                gt_changed_indices.add(gt_idx)
                gt_cells_filled += 1

        # --- Researchers → EXT_Project lead (PI) + EXT_Collaborator names ---
        researchers_val = dim_row.get('Researchers')
        if not _is_empty(researchers_val):  # only proceed if Dimensions has researcher data
            names = [n.strip() for n in str(researchers_val).split(';') if n.strip()]  # split semicolon-separated names into a list
            if names:  # guard against an empty list after splitting
                pi_empty     = _is_empty(grants_tracker_data_edited.at[gt_idx, 'EXT_Project lead (PI)'])
                collab_empty = _is_empty(grants_tracker_data_edited.at[gt_idx, 'EXT_Collaborator names'])

                if pi_empty:  # PI field is blank — fill first researcher into PI, rest into Collaborators
                    grants_tracker_data_edited.at[gt_idx, 'EXT_Project lead (PI)'] = names[0]
                    gt_changed_indices.add(gt_idx)
                    gt_cells_filled += 1
                    if collab_empty and len(names) > 1:  # only fill Collaborators if it's also blank and there are additional names
                        grants_tracker_data_edited.at[gt_idx, 'EXT_Collaborator names'] = '; '.join(names[1:])
                        gt_cells_filled += 1
                elif collab_empty and len(names) > 1:  # PI is already filled but Collaborators is blank — add all names except the first (assumed to be the PI)
                    grants_tracker_data_edited.at[gt_idx, 'EXT_Collaborator names'] = '; '.join(names[1:])
                    gt_changed_indices.add(gt_idx)
                    gt_cells_filled += 1

# Remove matched rows from dimensions_data — they already exist in grants_tracker_data
before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(gt_matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

### 22 grants matched and removed from Dimensions data - same as Stella ###

print(f"Matched {len(gt_matched_ids)} grants with grants_tracker_data")
print(f"Filled {gt_cells_filled} missing values across {len(gt_changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 22 grants with grants_tracker_data
Filled 31 missing values across 22 rows
Removed 22 rows from dimensions_data (173 remaining)


In [15]:
# Inspect changes: all columns, only rows that were modified, changed cells highlighted green.

gt_changes_view = grants_tracker_data_edited.loc[sorted(gt_changed_indices)]

def _highlight_filled_gt(data):
    """Green background on cells that were empty in the original but filled in the edited version."""
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        for col in data.columns:
            if _is_empty(grants_tracker_data.at[idx, col]) and not _is_empty(data.at[idx, col]):
                styles.at[idx, col] = 'background-color: #c6efce; color: #276221'
    return styles

print(f"{len(gt_changed_indices)} rows modified")
gt_styled = gt_changes_view.style.apply(_highlight_filled_gt, axis=None)
gt_styled.to_excel("1_deduplication/data_changes/dimensions_changes_gt_data.xlsx", index=True)
print("Saved → 1_deduplication/data_changes/dimensions_changes_gt_data.xlsx")
gt_changes_view

22 rows modified
Saved → 1_deduplication/data_changes/dimensions_changes_gt_data.xlsx


,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_2035 expenditures (#),INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags
446,Developing an\n industry-leading bioactive TG...,765480,NaN,GBP,968059,NaN,Awarded,https://gtr.ukri.org/projects?ref=83017161,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,Qkine combines proprietary animal-free biomanu...,grant.14950960,NaN,NaN,NaN
447,Up-cycling of whey streams via microalgae\n i...,518809,NaN,CHF,582439,NaN,Awarded,https://www.aramis.admin.ch/Grunddaten/?Projec...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,"Whey2blue, through the upcycling of dairy side...",grant.14934416,NaN,NaN,NaN
448,Plant-Based Proteins and Antioxidants in\n Hy...,209915,NaN,EUR,222771,NaN,Awarded,https://cordis.europa.eu/project/id/101206095,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,Dysphagia is a common condition among the elde...,grant.14932541,NaN,NaN,NaN
452,Transforming Agri-Food By-Products into\n Hig...,2258016,NaN,EUR,2396433,NaN,Awarded,https://cordis.europa.eu/project/id/101217636,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,MOA Foodtech: Pioneering Sustainable Food Solu...,grant.14955227,NaN,NaN,NaN
453,Harnessing the immense potential of\n precisi...,2485840,NaN,EUR,2638230,NaN,Awarded,https://ec.europa.eu/info/funding-tenders/oppo...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,Melt&Marble (M&M) is a Sweden-based company th...,grant.14917504,NaN,NaN,NaN
454,Discovery platform for novel thermostable\n r...,341713,NaN,GBP,432140,NaN,Awarded,https://gtr.ukri.org/projects?ref=10146550,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,This 18-month Industrial Research project will...,grant.14950915,NaN,NaN,NaN
456,Edible Soft Matter,267922,NaN,GBP,338807,NaN,Awarded,https://gtr.ukri.org/projects?ref=EP%2FU537044...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,Food is essential to life. People have become ...,grant.14950107,NaN,NaN,NaN
457,MeatPrint - Exploring the mechanophysical\n a...,0,NaN,NaN,0,NaN,Awarded,https://gepris.dfg.de/gepris/projekt/555001364...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,The project MeatPrint explores the mechanophys...,grant.14670240,NaN,NaN,NaN
458,Tackling the Astringency Enigma in\n Mouthfee...,202125,NaN,EUR,214508,NaN,Awarded,https://cordis.europa.eu/project/id/101204137,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,An unhealthy diet is a major contributor to ch...,grant.14932344,NaN,NaN,NaN
459,An innovative whey protein powder that\n incl...,330989,NaN,GBP,418581,NaN,Awarded,https://gtr.ukri.org/projects?ref=10147176,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,GoFitt is a UK-based foodtech SME led by Milen...,grant.14950431,NaN,NaN,NaN


### 4. Grants Tracker data vs last year's data

#### a. Dimensions ID match

In [16]:
### Match grants_tracker_data_edited against last_year_data_edited using Dimensions ID ###

# Snapshot before this step so the inspection cell can highlight only what THIS step changed.
last_year_data_pre_gt = last_year_data_edited.copy()

gt_lyd_col_map = {
    'EXT_Title':                              'Title',
    'EXT_Abstract':                           'Abstract',
    'INT_Total amount (actual currency)':     'Total amount',
    'INT_Gov contribution (actual currency)': 'Gov contribution',
    'INT_Currency type':                      'Currency',
    'EXT_Total amount (USD)':                 'Total amount (USD)',
    'EXT_Gov contribution (USD)':             'Gov contribution (USD)',
    'EXT_Funding decision':                   'Funding decision',
    'EXT_Project start date (estimated)':     'Project start date',
    'INT_End date':                           'End date',
    'EXT_Project lead (PI)':                  'Project lead (PI)',
    'EXT_PI organization':                    'PI organisation',
    'EXT_PI organization type':               'PI organisation type',
    'EXT_PI organization country':            'PI organisation country',
    'EXT_PI organization region':             'PI organisation region',
    'PI organisation state':                  'PI organisation state',
    'EXT_Collaborator names':                 'Collaborator names',
    'EXT_Collaborator organizations':         'Collaborator institutions',
    'EXT_Funder name':                        'Funder name',
    'EXT_Funder type':                        'Funder type',
    'EXT_Funder country':                     'Funder Country',
    'EXT_Funder region':                      'Funder region',
    'EXT_Production platform':                'Production platform',
    'EXT_End product type':                   'End product type',
    'EXT_Award purpose':                      'Award purpose',
    'EXT_URL for announcement':               'URL for announcement',
    'EXT_Date added':                         'Date added',
    'EXT_Last modified':                      'Last modified',
    'EXT_Years project starts':               'Year project started',
    'INT_GFI grantee?':                       'GFI grantee',
    'INT_GFI Los?':                           'GFI LOS',
    'INT_Link to Los':                        'Link to LOS',
    'INT_GFI partner?':                       'GFI partner',
    'INT_Tier':                               'Tier'

}

# Funding columns where a 0 value is treated as missing and can be overwritten
funding_cols_lyd2 = {'Total amount', 'Total amount (USD)', 'Gov contribution', 'Gov contribution (USD)'}

# Build lookup: Identification code → row indices in last_year_data_edited
lyd2_id_map = {}
for idx, val in last_year_data_edited['Identification code'].items():  # loop over every row in last_year_data_edited
    if not _is_empty(val):  # skip rows with no Identification code
        lyd2_id_map.setdefault(str(val).strip(), []).append(idx)

gt_lyd_matched_ids     = set()
gt_lyd_changed_indices = set()
gt_lyd_cells_filled    = 0

for _, gt_row in grants_tracker_data_edited.iterrows():  # loop over every grant in grants_tracker_data_edited
    grant_id = str(gt_row['Dimensions.ai grant ID']).strip()
    if grant_id not in lyd2_id_map:  # skip if this Dimensions ID doesn't appear in last_year_data_edited
        continue

    gt_lyd_matched_ids.add(grant_id)

    for lyd_idx in lyd2_id_map[grant_id]:  # loop over matching rows (usually just one)

        for gt_col, lyd_col in gt_lyd_col_map.items():  # loop over each column pair in the mapping
            if gt_col not in grants_tracker_data_edited.columns or lyd_col not in last_year_data_edited.columns:  # skip if either column doesn't exist
                continue
            gt_val = gt_row[gt_col]
            if _is_empty(gt_val):  # nothing to fill with
                continue
            target_val = last_year_data_edited.at[lyd_idx, lyd_col]
            overwrite_zero = lyd_col in funding_cols_lyd2 and _is_zero(target_val)  # for funding cols, treat 0 as fillable
            if _is_empty(target_val) or overwrite_zero:  # fill if empty, or if it's a 0 in a funding column
                last_year_data_edited.at[lyd_idx, lyd_col] = gt_val
                gt_lyd_changed_indices.add(lyd_idx)
                gt_lyd_cells_filled += 1

# Remove matched rows from grants_tracker_data_edited — they already exist in last_year_data_edited
before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['Dimensions.ai grant ID'].astype(str).str.strip().isin(gt_lyd_matched_ids)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)


### 182 matched grants and 187 rows removed - same as Stella, but also means there were 5 duplicate Dimensions IDs in the grants tracker ###
print(f"Matched {len(gt_lyd_matched_ids)} grants between grants_tracker_data_edited and last_year_data_edited")
print(f"Filled {gt_lyd_cells_filled} missing values across {len(gt_lyd_changed_indices)} rows")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")

Matched 182 grants between grants_tracker_data_edited and last_year_data_edited
Filled 491 missing values across 149 rows
Removed 187 rows from grants_tracker_data_edited (363 remaining)


In [17]:
# OPTIONAL - shows which grants were duplicated within the grants tracker data
dupes = grants_tracker_data[
    grants_tracker_data['Dimensions.ai grant ID'].isin(gt_lyd_matched_ids)
    & grants_tracker_data['Dimensions.ai grant ID'].duplicated(keep=False)
][['Dimensions.ai grant ID', 'EXT_Title']].sort_values('Dimensions.ai grant ID')
print(dupes)

    Dimensions.ai grant ID                                          EXT_Title
110         grant.12941157  CIRCular valorisation of industrial ALGAE wast...
393         grant.12941157  CIRCular\n  valorisation of industrial ALGAE w...
115         grant.13253023                                EIT Food Activities
440         grant.13253023                   EIT Food Activities (TASTE2MEAT)
5           grant.13879477  Fermentation optimisation for a palm oil alter...
372         grant.13879477  Fermentation optimisation for a palm oil alter...
218         grant.13909296  Harnessing genetic diversity of the novel rape...
405         grant.13909296  Harnessing genetic diversity of the novel\n  r...
78           grant.9965207  Novel texturized hybrid foods targeting future...
370          grant.9965207  Novel texturized\n  hybrid foods targeting fut...


In [18]:
# Inspect changes from this step only: compare last_year_data_edited against the pre-step snapshot.

gt_lyd_changes_view = last_year_data_edited.loc[sorted(gt_lyd_changed_indices)]

def _highlight_filled_gt_lyd(data):
    """Green background on cells that were empty before this step but filled by it."""
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        for col in data.columns:
            if _is_empty(last_year_data_pre_gt.at[idx, col]) and not _is_empty(data.at[idx, col]):
                styles.at[idx, col] = 'background-color: #c6efce; color: #276221'
    return styles

print(f"{len(gt_lyd_changed_indices)} rows modified")
gt_lyd_styled = gt_lyd_changes_view.style.apply(_highlight_filled_gt_lyd, axis=None)
gt_lyd_styled.to_excel("1_deduplication/data_changes/gt_changes_last_year_data.xlsx", index=True)
print("Saved → 1_deduplication/data_changes/gt_changes_last_year_data.xlsx")
gt_lyd_changes_view

149 rows modified
Saved → 1_deduplication/data_changes/gt_changes_last_year_data.xlsx


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
8,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,NaN,airtable,27600000,27600000,NOK,2604048.0,2604048.0,2401200.00,...,NaN,NaN,2023-04-21,2023-04-21,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>
60,"ECOnti - Accelerated, low ecological footprint...",Initial situation: \nMicrobial processes are e...,NaN,airtable,3600000,2700000,EUR,0.0,0.0,3600000.00,...,NaN,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>
135,Extruded and 3D printed vegan support structur...,No abstract,NaN,airtable,768825,768825,EUR,824278.0,824278.0,768825.00,...,NaN,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2022,<NA>
138,SUSTAINER: Sustainable production of structura...,Biomaterials are materials engineered to inter...,NaN,airtable,750000,750000,EUR,0.0,0.0,750000.00,...,NaN,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>
163,Surface-active algae protein isolates for vega...,No abstract,NaN,airtable,521407,521407,EUR,563410.0,563410.0,521407.00,...,NaN,NaN,2025-01-14,2025-05-02,NaN,NaN,NaN,NaN,2023,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933,A novel support material for 3D bioprinting an...,"""Three-dimensional (3D) bioprinting holds grea...",A novel support material for 3D bioprinting an...,Dimensions,150000,150000,EUR,160500.0,160371.0,150000.00,...,NaN,NaN,2025-01-14,2026-03-17,"""Three-dimensional (3D) bioprinting holds grea...",NaN,NaN,NaN,2022,2023
934,Whole-cut cell-based fish fillet production by...,NaN,Whole-cut cell-based fish fillet production by...,Dimensions,20590,20590,EUR,22221.0,22221.0,NaN,...,NaN,NaN,2025-01-14,2026-03-17,NaN,NaN,NaN,NaN,2022,2026
936,Advanced Cellular Hierarchical Tissue-Imitatio...,ACHIEVE focuses on the application of Excluded...,Advanced Cellular Hierarchical Tissue-Imitatio...,Dimensions,2076770,2076770,EUR,2233939.0,2246117.0,2076770.00,...,NaN,NaN,2025-01-14,2026-03-17,ACHIEVE focuses on the application of Excluded...,NaN,NaN,NaN,2021,2026
938,3D Cell Scaffolds for Clean Meat Cultivation,Conventional farming of animal-based protein h...,3D Cell Scaffolds for Clean Meat Cultivation,Dimensions,230616,230616,CHF,260104.0,168076.0,246759.12,...,NaN,NaN,2025-01-14,2026-03-17,Conventional farming of animal-based protein h...,NaN,NaN,NaN,2023,2024


#### b. Title-based match

In [ ]:
### Find title-based matches between grants_tracker_data_edited and last_year_data_edited ###
# Full exact match on normalised title (stripped, lowercased).
# Grants tracker already had Dimensions-ID matches removed, so any title match here
# is a grant that exists in both datasets without a shared Dimensions ID.

def _norm_title(val):
    if _is_empty(val):
        return None
    return str(val).strip().lower()

# Build lookup: normalised title → row indices in last_year_data_edited
lyd_title_map = {}
for idx, val in last_year_data_edited['Title'].items():
    norm = _norm_title(val)
    if norm:
        lyd_title_map.setdefault(norm, []).append(idx)

title_match_rows = []

for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    norm = _norm_title(gt_row.get('EXT_Title'))
    if norm and norm in lyd_title_map:
        for lyd_idx in lyd_title_map[norm]:
            title_match_rows.append({
                'GT index':                gt_idx,
                'LYD index':               lyd_idx,
                'Title':                   gt_row.get('EXT_Title'),
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'GT Funder':               gt_row.get('EXT_Funder name'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
                'GT Start date':           gt_row.get('EXT_Project start date (estimated)'),
                'LYD Start date':          last_year_data_edited.at[lyd_idx, 'Project start date'],
                'GT End date':             gt_row.get('INT_End date'),
                'LYD End date':            last_year_data_edited.at[lyd_idx, 'End date'],
            })

title_matches_df = pd.DataFrame(title_match_rows)

unique_gt = title_matches_df['GT index'].nunique() if len(title_matches_df) else 0
print(f"{len(title_matches_df)} (GT, LYD) pairs found → {unique_gt} unique GT rows will be removed")

# Highlight any GT rows that match more than one LYD row (title exists multiple times in last_year_data)
multi = title_matches_df[title_matches_df.duplicated('GT index', keep=False)]
if len(multi):
    print(f"\nNote: {multi['GT index'].nunique()} GT row(s) match multiple LYD rows (duplicate title in last_year_data):")
    print(multi[['GT index', 'LYD index', 'Title']].to_string(index=False))


title_matches_df.to_excel("1_deduplication/data_changes/title_matches_review.xlsx", index=False)
print("Saved -> 1_deduplication/data_changes/title_matches_review.xlsx")
title_matches_df

13 (GT, LYD) pairs found → 10 unique GT rows will be removed

Note: 3 GT row(s) match multiple LYD rows (duplicate title in last_year_data):
 GT index  LYD index                                                                       Title
      249        708                                           Sustainable Proteins from Seaweed
      249        976                                           Sustainable Proteins from Seaweed
      259        717                Meat replacement and systems of edibility in Asia and beyond
      259       1122                Meat replacement and systems of edibility in Asia and beyond
      313        127 EAGLE: Enhanced Analytical and Genetics Tools for Improving UK Food Legumes
      313        288 EAGLE: Enhanced Analytical and Genetics Tools for Improving UK Food Legumes


,GT index,LYD index,Title,GT Dimensions ID,LYD Identification code,GT Funder,LYD Funder,GT Total (USD),LYD Total (USD),GT Start date,LYD Start date,GT End date,LYD End date
0,7,1120,SusKelpFood – Sustainable ingredients from cul...,grant.9965162,NaN,The Research Council of Norway,Research Council of Norway,2723733.0,NaN,NaT,NaT,None,<NA>
1,237,697,Impact analysis of the novel food knowledge unit,NaN,NaN,SA Tallinna Teaduspark Tehnopol,Tallinn Technology Park (Tehnopol),26532.0,26532.0,2024-01-23,2024-01-23,None,<NA>
2,238,698,Microbial proteins as food product?,NaN,NaN,Federal Ministry of Food and Agriculture,Federal Ministry of Food and Agriculture (BMEL),NaN,NaN,2024-06-03,2024-06-03,None,<NA>
3,239,699,Food4Cells Sustainable cell culture media for ...,NaN,NaN,Research Council of Norway and participating c...,Research Council of Norway,338498.0,338498.0,2025-01-01,2025-01-01,None,<NA>
4,249,708,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,NaT,None,<NA>
5,249,976,Sustainable Proteins from Seaweed,NaN,NaN,NaN,Research Foundation – Flanders (FWO),NaN,NaN,NaT,2024-01-10,None,<NA>
6,259,717,Meat replacement and systems of edibility in A...,NaN,grant.14751793,Research Council of Norway,Research Council of Norway,1127788.0,1127788.0,2025-01-01,2025-01-01,None,2029
7,259,1122,Meat replacement and systems of edibility in A...,NaN,NaN,Research Council of Norway,Research Council of Norway,1127788.0,NaN,2025-01-01,NaT,None,<NA>
8,313,127,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,grant.12930078,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,1048464.0,2025-01-22,2023-01-01,None,<NA>
9,313,288,EAGLE: Enhanced Analytical and Genetics Tools ...,grant.14049012,NaN,Federal Department of Economic Affairs Educati...,UK Research and Innovation (UKRI),283485.0,274575.0,2025-01-22,2022-01-01,None,<NA>


In [20]:
### Gap-fill and removal for title-based matches ###

############################################################################################
# Add GT index values to this set to skip specific rows — they will not be gap-filled
# or removed from grants_tracker_data_edited.
excluded_gt_indices = set()
# e.g. excluded_gt_indices = {5, 12, 34}
############################################################################################

last_year_data_pre_title = last_year_data_edited.copy()

title_matched_gt_indices  = set()
title_changed_indices     = set()
title_cells_filled        = 0

for _, match in title_matches_df.iterrows():
    gt_idx  = int(match['GT index'])
    lyd_idx = int(match['LYD index'])

    if gt_idx in excluded_gt_indices:
        continue

    title_matched_gt_indices.add(gt_idx)
    gt_row = grants_tracker_data_edited.loc[gt_idx]

    for gt_col, lyd_col in gt_lyd_col_map.items():
        if gt_col not in grants_tracker_data_edited.columns or lyd_col not in last_year_data_edited.columns:
            continue
        gt_val = gt_row[gt_col]
        if _is_empty(gt_val):
            continue
        target_val = last_year_data_edited.at[lyd_idx, lyd_col]
        overwrite_zero = lyd_col in funding_cols_lyd2 and _is_zero(target_val)
        if _is_empty(target_val) or overwrite_zero:
            last_year_data_edited.at[lyd_idx, lyd_col] = gt_val
            title_changed_indices.add(lyd_idx)
            title_cells_filled += 1

# Remove matched (non-excluded) rows from grants_tracker_data_edited
before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited.index.isin(title_matched_gt_indices)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)

### 13 matched grants, but 3 duplicate titles in LYD, so 10 removed from grants_tracker_data_edited - Stella said (n=13) - not sure if removed or matched ###

print(f"Processed {len(title_matched_gt_indices)} title matches ({len(excluded_gt_indices)} excluded)")
print(f"Filled {title_cells_filled} missing values across {title_changed_indices.__len__()} rows")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")

Processed 10 title matches (0 excluded)
Filled 22 missing values across 7 rows
Removed 10 rows from grants_tracker_data_edited (353 remaining)


In [21]:
# Inspect changes from this step only: compare last_year_data_edited against the pre-step snapshot.

title_changes_view = last_year_data_edited.loc[sorted(title_changed_indices)]

def _highlight_filled_title(data):
    """Green background on cells that were empty before this step but filled by it."""
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        for col in data.columns:
            if _is_empty(last_year_data_pre_title.at[idx, col]) and not _is_empty(data.at[idx, col]):
                styles.at[idx, col] = 'background-color: #c6efce; color: #276221'
    return styles

print(f"{len(title_changed_indices)} rows modified")
title_styled = title_changes_view.style.apply(_highlight_filled_title, axis=None)
title_styled.to_excel("1_deduplication/data_changes/gt_title-match_changes_last_year_data.xlsx", index=True)
print("Saved → 1_deduplication/data_changes/gt_title-match_changes_last_year_data.xlsx")
title_changes_view

7 rows modified
Saved → 1_deduplication/data_changes/gt_title-match_changes_last_year_data.xlsx


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
288,EAGLE: Enhanced Analytical and Genetics Tools ...,Despite providing an excellent source of high-...,NaN,airtable,261500,209235,GBP,274575.0,219697.0,311185.00,...,NaN,NaN,2024-03-26,2025-05-02,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022,<NA>
553,Combinatorial solidification approaches for th...,NaN,NaN,airtable,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2024-10-31,2024-10-31,NaN,NaN,NaN,NaN,2023,<NA>
976,Sustainable Proteins from Seaweed,Because of the generally high standard of livi...,NaN,FRIS,NaN,NaN,EUR,NaN,NaN,NaN,...,NaN,NaN,2025-04-14,2025-04-14,NaN,NaN,NaN,NaN,2024,<NA>
1001,Occurrence and behavior of spore-forming bacte...,NaN,Vorkommen und Verhalten sporenbildender Bakter...,FEI,524719,524719,EUR,NaN,NaN,524719.00,...,NaN,NaN,2025-10-06,2026-03-17,NaN,NaN,NaN,NaN,2023,<NA>
1104,Continuous fermentation of milk protein (Conti...,"Over the past decade, there has been an increa...",NaN,groenprojektbank.dk/,5682434,2557095.3,DKK,969452.0,NaN,738716.42,...,NaN,NaN,2025-12-17,2025-12-17,NaN,NaN,NaN,NaN,2024,<NA>
1120,SusKelpFood – Sustainable ingredients from cul...,There is a growing need to produce more food i...,NaN,https://prosjektbanken.forskningsradet.no/,26800000,26800000,NOK,2723733.0,2723733.0,2331600.00,...,NaN,NaN,2025-01-14,2026-03-17,NaN,NaN,NaN,NaN,2021,<NA>
1122,Meat replacement and systems of edibility in A...,Alternative proteins can replace meat. Given t...,NaN,https://prosjektbanken.forskningsradet.no/,12000000,12000000,NOK,1127788.0,NaN,1044000.00,...,NaN,NaN,2025-04-28,2025-04-28,NaN,NaN,NaN,NaN,2025,<NA>


#### c. Partial title-based match

In [22]:
### Find partial title-based matches between grants_tracker_data_edited and last_year_data_edited ###
# Uses rapidfuzz token_sort_ratio: tokens are sorted before comparing, so word-order
# differences score highly. Threshold 85 gives tight matches; lower it to surface more.
# Unlike the Excel SEARCH formula (substring containment only), this also handles
# truncations, minor rewordings, and transposed words.

try:
    from rapidfuzz import fuzz as _rfuzz
except ImportError:
    raise ImportError("rapidfuzz not found — run: conda install -c conda-forge rapidfuzz")

SIMILARITY_THRESHOLD = 85  # 0-100; raise to tighten, lower to surface more candidates

def _norm_partial(val):
    if _is_empty(val):
        return None
    return str(val).strip().lower()

# Pre-build the LYD title list once so the inner loop is fast
lyd_title_list = [
    (idx, _norm_partial(val))
    for idx, val in last_year_data_edited['Title'].items()
    if not _is_empty(val)
]

partial_match_rows = []

for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    gt_norm = _norm_partial(gt_row.get('EXT_Title'))
    if not gt_norm:
        continue
    for lyd_idx, lyd_norm in lyd_title_list:
        score = _rfuzz.token_sort_ratio(gt_norm, lyd_norm)
        if score >= SIMILARITY_THRESHOLD:
            partial_match_rows.append({
                'GT index':                gt_idx,
                'LYD index':               lyd_idx,
                'Similarity':              score,
                'GT Title':                gt_row.get('EXT_Title'),
                'LYD Title':               last_year_data_edited.at[lyd_idx, 'Title'],
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'GT Funder':               gt_row.get('EXT_Funder name'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
                'GT Start date':           gt_row.get('EXT_Project start date (estimated)'),
                'LYD Start date':          last_year_data_edited.at[lyd_idx, 'Project start date'],
                'GT End date':             gt_row.get('INT_End date'),
                'LYD End date':            last_year_data_edited.at[lyd_idx, 'End date'],
            })

partial_matches_df = pd.DataFrame(partial_match_rows).sort_values('Similarity', ascending=False).reset_index(drop=True)

unique_gt = partial_matches_df['GT index'].nunique() if len(partial_matches_df) else 0
print(f"{len(partial_matches_df)} (GT, LYD) pairs found at threshold {SIMILARITY_THRESHOLD} → {unique_gt} unique GT rows will be removed")

multi = partial_matches_df[partial_matches_df.duplicated('GT index', keep=False)]
if len(multi):
    print(f"\nNote: {multi['GT index'].nunique()} GT row(s) match multiple LYD rows:")
    print(multi[['GT index', 'LYD index', 'Similarity', 'GT Title', 'LYD Title']].to_string(index=False))


partial_matches_df.to_excel("1_deduplication/data_changes/gt_title-partial-match_review.xlsx", index=False)
print("Saved → 1_deduplication/data_changes/gt_title-partial-match_review.xlsx")
partial_matches_df

72 (GT, LYD) pairs found at threshold 85 → 30 unique GT rows will be removed

Note: 13 GT row(s) match multiple LYD rows:
 GT index  LYD index  Similarity                                                                                                                                                     GT Title                                                                                                                                              LYD Title
      234        695  100.000000                                                                Ecological critique and civic experiments in plant-based\n  agricultural alternatives (CIVEX)                                                             Ecological critique and civic experiments in plant-based agricultural alternatives (CIVEX)
      235        963  100.000000                Revealing factors determining quality and functionality of plant\n  proteins fractionated by sustainable dry processing for human consumption       

,GT index,LYD index,Similarity,GT Title,LYD Title,GT Dimensions ID,LYD Identification code,GT Funder,LYD Funder,GT Total (USD),LYD Total (USD),GT Start date,LYD Start date,GT End date,LYD End date
0,234,695,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,NaN,Independent Research Fund Denmark,Independent Research Fund Denmark,314570.0,314570.00,2024-02-01,2024-02-01,None,<NA>
1,235,963,100.000000,Revealing factors determining quality and func...,Revealing factors determining quality and func...,NaN,3164-00216A,Independent Research Fund Denmark,Independent Research Fund Denmark,305004.0,305003.55,2024-09-11,2024-09-11,None,<NA>
2,235,696,100.000000,Revealing factors determining quality and func...,Revealing factors determining quality and func...,NaN,NaN,Independent Research Fund Denmark,Independent Research Fund Denmark,305004.0,305004.00,2024-09-11,2024-09-11,None,<NA>
3,234,962,100.000000,Ecological critique and civic experiments in p...,Ecological critique and civic experiments in p...,NaN,3164-00027A,Independent Research Fund Denmark,Independent Research Fund Denmark,314570.0,314569.51,2024-02-01,2024-02-01,None,<NA>
4,236,822,100.000000,Socio-ecological research to understand the po...,Socio-ecological research to understand the po...,NaN,grant.14682509,ESRC,UK Research and Innovation (UKRI),0.0,0.00,2024-09-30,2024-09-30,None,2028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,158,680,93.023256,Engineering safe faba beans by targeting the a...,Engineering safe faba beans by targeting the a...,grant.13984174,"grant.13984174, grant.13909180",European Commission,European Commission,232244.0,232655.00,NaT,2024-11-01,None,<NA>
68,264,671,90.657439,Harnessing the immense potential of\n precisi...,Harnessing the immense potential of precision ...,grant.14917504,grant.14613568,European Commission,European Commission,2638230.0,2690896.00,2025-01-01,2025-01-01,None,2026
69,308,882,90.000000,Plant breeding research P3 joint project: 'Rap...,Plant Breeding Research P2 joint project: 'Rap...,grant.13246253,grant.9063165,Federal Ministry of Education and Research (BMBF),Federal Ministry of Education and Research (BMBF),48759.0,378372.00,2025-10-20,2020-03-01,None,2023
70,63,522,87.619048,Ecologically sustainable food for the elderly ...,Ecologically sustainable food for obese elderly,grant.12908252,grant.9549928,Dutch Research Council (NWO),Dutch Research Council (NWO),0.0,0.00,2022-09-01,2022-02-01,None,<NA>


In [23]:
### Remove partial title matches from grants_tracker_data_edited ###
# Gap-filling is intentionally skipped — partial title matches are not reliable
# enough to fill data across. Review partial_title_matches_review.xlsx manually
# and add any GT index values that are legitimately new data to the exclusion set.

############################################################################################
# MUST REVIEW THE PREVIOUS EXCEL DOC TO IDENTIFY WHICH GT ENTRIES ARE LEGITIMATE NEW DATA AND SHOULD BE EXCLUDED FROM REMOVAL.
# Add GT index values here to skip — they will NOT be removed from grants_tracker_data_edited.
excluded_partial_gt_indices = set()
excluded_partial_gt_indices = {63, 264, 274, 307, 308, 309}
############################################################################################

partial_matched_gt_indices = set()
for _, match in partial_matches_df.iterrows():
    gt_idx = int(match['GT index'])
    if gt_idx not in excluded_partial_gt_indices:
        partial_matched_gt_indices.add(gt_idx)

before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited.index.isin(partial_matched_gt_indices)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)

print(f"Processed {len(partial_matched_gt_indices)} partial matches ({len(excluded_partial_gt_indices)} excluded)")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")
grants_tracker_data_edited.head()

Processed 24 partial matches (6 excluded)
Removed 24 rows from grants_tracker_data_edited (329 remaining)


,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_2035 expenditures (#),INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags
0,Food processing residues to climate smart food...,800000,800000,SEK,76376,76376.0,NaN,https://www.vr.se/swecris.html#/project/2022-0...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.4456273,NaN,BF,NaN
1,MET2FOOD,108738,108738,GBP,139436,139436.0,NaN,https://gtr.ukri.org/projects?ref=91600,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.9555967,NaN,BF,NaN
2,Recovery of value added ingredients from the w...,132001,132001,GBP,206365,206365.0,NaN,https://gtr.ukri.org/projects?ref=101401,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.3843264,NaN,PF,NaN
3,Novel yeast-based biotechnological route for t...,2398095,0,DKK,347576,0.0,NaN,https://novonordiskfonden.dk/app/uploads/NNF-g...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.13909923,NaN,PF,NaN
4,Tree to Feed - Circular production of sustaina...,8511411,8511411,SEK,816675,816675.0,NaN,https://www.vr.se/swecris.html#/project/2022-0...,NaN,NaN,...,NaN,ERROR,NaN,NaN,NaN,NaN,grant.12926671,NaN,BF,NaN


#### d. Append new grants tracker data to last year's data

In [ ]:
# Extra mappings for append — columns not needed for gap-filling but required when
# adding new rows: Dimensions ID → Identification code.
append_extra_map = {
    'Dimensions.ai grant ID':                 'Identification code',
}

# Merge gt_lyd_col_map with the extra mappings (extra_map wins on any overlap)
full_append_map = {**gt_lyd_col_map, **append_extra_map}

gt_for_append = grants_tracker_data_edited.rename(columns={
    gt_col: lyd_col for gt_col, lyd_col in full_append_map.items()
})

# Keep only columns that exist in last_year_data_edited
gt_for_append = gt_for_append[[c for c in gt_for_append.columns if c in last_year_data_edited.columns]].copy()

# Tag all appended rows as coming from the grants tracker
gt_for_append['Database'] = 'airtable'

last_year_data_edited = pd.concat([last_year_data_edited, gt_for_append], ignore_index=True)

print(f"Appended {len(grants_tracker_data_edited)} rows from grants_tracker_data_edited")
print(f"Total rows in last_year_data_edited: {len(last_year_data_edited)}")

last_year_data_edited.head()

Appended 329 rows from grants_tracker_data_edited
Total rows in last_year_data_edited: 1509


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,2023-04-13,2023-12-04 00:00:00,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,NaN,NaN,2023-02-27,2023-04-19 00:00:00,NaN,NaN,NaN,Tier 4 (No GFI involvement),2022,<NA>
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,2024-03-26,2025-05-02 00:00:00,NaN,NaN,NaN,Tier 4 (No GFI involvement),2023,<NA>
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,2023-02-28,2023-04-20 00:00:00,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,NaN,NaN,2023-04-04,2023-04-19 00:00:00,NaN,NaN,NaN,Tier 4 (No GFI involvement),2021,<NA>


### Dimensions title matching

#### a. Full title match

In [ ]:
### Find full title matches between dimensions_data and last_year_data_edited ###
# Matches on 'Title translated' in dimensions_data against 'Title' in last_year_data_edited.
# Exact match after stripping and lowercasing.

# Build lookup: normalised title -> row indices in last_year_data_edited
lyd_dim_title_map = {}
for idx, val in last_year_data_edited['Title'].items():
    norm = _norm_title(val)
    if norm:
        lyd_dim_title_map.setdefault(norm, []).append(idx)

dim_title_match_rows = []

for dim_idx, dim_row in dimensions_data.iterrows():
    norm = _norm_title(dim_row.get('Title translated'))
    if norm and norm in lyd_dim_title_map:
        for lyd_idx in lyd_dim_title_map[norm]:
            dim_title_match_rows.append({
                'Dim index':               dim_idx,
                'LYD index':               lyd_idx,
                'Title':                   dim_row.get('Title translated'),
                'Dim Grant ID':            dim_row.get('Grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Funding amount in USD'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LYD Start date':          last_year_data_edited.at[lyd_idx, 'Project start date'],
                'Dim End Year':            dim_row.get('End Year'),
                'LYD End date':            last_year_data_edited.at[lyd_idx, 'End date'],
            })

dim_title_matches_df = pd.DataFrame(dim_title_match_rows)

unique_dim = dim_title_matches_df['Dim index'].nunique() if len(dim_title_matches_df) else 0
print(f"{len(dim_title_matches_df)} (Dim, LYD) pairs found -> {unique_dim} unique Dimensions rows will be removed")

multi = dim_title_matches_df[dim_title_matches_df.duplicated('Dim index', keep=False)]
if len(multi):
    print(f"\nNote: {multi['Dim index'].nunique()} Dimensions row(s) match multiple LYD rows (duplicate title in last_year_data):")
    print(multi[['Dim index', 'LYD index', 'Title']].to_string(index=False))


dim_title_matches_df.to_excel("1_deduplication/data_changes/dim_title_matches_review.xlsx", index=False)
print("Saved -> 1_deduplication/data_changes/dim_title_matches_review.xlsx")
dim_title_matches_df

1 (Dim, LYD) pairs found -> 1 unique Dimensions rows will be removed
Saved -> 1_deduplication/dim_title_matches_review.xlsx


,Dim index,LYD index,Title,Dim Grant ID,LYD Identification code,Dim Funder,LYD Funder,Dim Total (USD),LYD Total (USD),Dim Start date,LYD Start date,Dim End Year,LYD End date
0,38,507,BiochAIn - BiochAIn is a software project for ...,grant.14970420,grant.13728161,Austrian Research Promotion Agency,Austrian Research Promotion Agency (FFG),0,0.0,2025-01-04,2023-01-03,2026.0,<NA>


In [26]:
### Gap-fill and removal for Dimensions full title matches ###

############################################################################################
# Add Dim index values here to skip — they will not be gap-filled or removed.
excluded_dim_title_indices = set()
excluded_dim_title_indices = {38}
############################################################################################

last_year_data_pre_dim_title = last_year_data_edited.copy()

dim_title_matched_indices = set()
dim_title_changed_indices = set()
dim_title_cells_filled    = 0

for _, match in dim_title_matches_df.iterrows():
    dim_idx = int(match['Dim index'])
    lyd_idx = int(match['LYD index'])

    if dim_idx in excluded_dim_title_indices:
        continue

    dim_title_matched_indices.add(dim_idx)
    dim_row = dimensions_data.loc[dim_idx]

    for dim_col, lyd_col in col_map.items():
        if dim_col not in dimensions_data.columns or lyd_col not in last_year_data_edited.columns:
            continue
        dim_val = dim_row[dim_col]
        if _is_empty(dim_val):
            continue
        target_val = last_year_data_edited.at[lyd_idx, lyd_col]
        overwrite_zero = lyd_col in funding_cols_lyd and _is_zero(target_val)
        if _is_empty(target_val) or overwrite_zero:
            last_year_data_edited.at[lyd_idx, lyd_col] = dim_val
            dim_title_changed_indices.add(lyd_idx)
            dim_title_cells_filled += 1

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data.index.isin(dim_title_matched_indices)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Processed {len(dim_title_matched_indices)} title matches ({len(excluded_dim_title_indices)} excluded)")
print(f"Filled {dim_title_cells_filled} missing values across {len(dim_title_changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Processed 0 title matches (1 excluded)
Filled 0 missing values across 0 rows
Removed 0 rows from dimensions_data (173 remaining)


In [ ]:
# Inspect changes from this step only: compare last_year_data_edited against the pre-step snapshot.

dim_title_changes_view = last_year_data_edited.loc[sorted(dim_title_changed_indices)]

def _highlight_filled_dim_title(data):
    """Green background on cells that were empty before this step but filled by it."""
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for idx in data.index:
        for col in data.columns:
            if _is_empty(last_year_data_pre_dim_title.at[idx, col]) and not _is_empty(data.at[idx, col]):
                styles.at[idx, col] = 'background-color: #c6efce; color: #276221'
    return styles

print(f"{len(dim_title_changed_indices)} rows modified")
dim_title_styled = dim_title_changes_view.style.apply(_highlight_filled_dim_title, axis=None)
dim_title_styled.to_excel("1_deduplication/data_changes/dim_title_changes_last_year_data.xlsx", index=True)
print("Saved -> 1_deduplication/data_changes/dim_title_changes_last_year_data.xlsx")
dim_title_changes_view

0 rows modified
Saved -> 1_deduplication/dim_title_changes_last_year_data.xlsx


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,APP site,Minority serving institution,Date added,Last modified,GFI LOS,Link to LOS,GFI partner,Tier,Year project started,End date


#### b. Partial title match

In [ ]:
### Find partial title matches between dimensions_data and last_year_data_edited ###
# Uses rapidfuzz token_sort_ratio on 'Title translated' vs last_year_data 'Title'.

try:
    from rapidfuzz import fuzz as _rfuzz
except ImportError:
    raise ImportError("rapidfuzz not found - run: conda install -c conda-forge rapidfuzz")

DIM_SIMILARITY_THRESHOLD = 85  # 0-100; raise to tighten, lower to surface more candidates

# Pre-build the LYD title list once
lyd_dim_partial_list = [
    (idx, _norm_partial(val))
    for idx, val in last_year_data_edited['Title'].items()
    if not _is_empty(val)
]

dim_partial_match_rows = []

for dim_idx, dim_row in dimensions_data.iterrows():
    dim_norm = _norm_partial(dim_row.get('Title translated'))
    if not dim_norm:
        continue
    for lyd_idx, lyd_norm in lyd_dim_partial_list:
        score = _rfuzz.token_sort_ratio(dim_norm, lyd_norm)
        if score >= DIM_SIMILARITY_THRESHOLD:
            dim_partial_match_rows.append({
                'Dim index':               dim_idx,
                'LYD index':               lyd_idx,
                'Similarity':              score,
                'Dim Title':               dim_row.get('Title translated'),
                'LYD Title':               last_year_data_edited.at[lyd_idx, 'Title'],
                'Dim Grant ID':            dim_row.get('Grant ID'),
                'LYD Identification code': last_year_data_edited.at[lyd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LYD Funder':              last_year_data_edited.at[lyd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Funding amount in USD'),
                'LYD Total (USD)':         last_year_data_edited.at[lyd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LYD Start date':          last_year_data_edited.at[lyd_idx, 'Project start date'],
                'Dim End Year':            dim_row.get('End Year'),
                'LYD End date':            last_year_data_edited.at[lyd_idx, 'End date'],
            })

dim_partial_matches_df = pd.DataFrame(dim_partial_match_rows).sort_values('Similarity', ascending=False).reset_index(drop=True)

unique_dim_p = dim_partial_matches_df['Dim index'].nunique() if len(dim_partial_matches_df) else 0
print(f"{len(dim_partial_matches_df)} (Dim, LYD) pairs found at threshold {DIM_SIMILARITY_THRESHOLD} -> {unique_dim_p} unique Dimensions rows will be removed")

multi_p = dim_partial_matches_df[dim_partial_matches_df.duplicated('Dim index', keep=False)]
if len(multi_p):
    print(f"\nNote: {multi_p['Dim index'].nunique()} Dimensions row(s) match multiple LYD rows:")
    print(multi_p[['Dim index', 'LYD index', 'Similarity', 'Dim Title', 'LYD Title']].to_string(index=False))

### I only got 6 partial matches while Stella got 73 ###

dim_partial_matches_df.to_excel("1_deduplication/data_changes/dim_partial_title_matches_review.xlsx", index=False)
print("Saved -> 1_deduplication/data_changes/dim_partial_title_matches_review.xlsx")
dim_partial_matches_df

6 (Dim, LYD) pairs found at threshold 85 -> 6 unique Dimensions rows will be removed
Saved -> 1_deduplication/dim_partial_title_matches_review.xlsx


,Dim index,LYD index,Similarity,Dim Title,LYD Title,Dim Grant ID,LYD Identification code,Dim Funder,LYD Funder,Dim Total (USD),LYD Total (USD),Dim Start date,LYD Start date,Dim End Year,LYD End date
0,38,507,100.000000,BiochAIn - BiochAIn is a software project for ...,BiochAIn - BiochAIn is a software project for ...,grant.14970420,grant.13728161,Austrian Research Promotion Agency,Austrian Research Promotion Agency (FFG),0,0.0,2025-01-04,2023-01-03,2026.0,<NA>
1,37,1442,98.591549,"AlgFlavor, Enhancing algal biomass consumer ac...",AlgFlavor: Enhancing algal biomass consumer ac...,grant.14972633,NaN,The Research Council of Norway,Sustainable Blue Economy Partnership,259197,1904980.0,2025-01-01,2025-05-01,2028.0,<NA>
2,25,1444,96.078431,MICROALGAL BIOMASS VALORIZATION FOR SUSTAINABL...,BIOVAL: Microalgal biomass valorization for su...,grant.15162542,NaN,State Research Agency,Sustainable Blue Economy Partnership,225527,1904980.0,2025-01-01,2025-05-01,NaN,<NA>
3,5,1439,94.382022,Flexible and Efficient Capture and Bioconversi...,UNICO2RN: Flexible and Efficient Capture and B...,grant.14933116,NaN,European Commission,Horizon Europe Guarantee,7959724,8565440.0,2025-06-01,2025-06-01,2029.0,<NA>
4,8,1440,90.909091,Upcycling mushroom waste to replace animal der...,MYCOCIRCLE: Upcycling mushroom waste to replac...,grant.14933075,NaN,European Commission,Horizon Europe Guarantee,3700302,3983555.0,2025-09-01,2025-09-01,2028.0,<NA>
5,0,608,88.789238,Scientific Exchange to assess QUality and Risk...,Scientific Exchange to assess QUality and Risk...,grant.14917108,grant.14473519,European Commission,European Commission,1757503,1792594.0,2025-01-01,2025-01-01,2028.0,2028


In [29]:
### Remove partial title matches from dimensions_data ###
# Gap-filling is intentionally skipped — partial title matches are not reliable
# enough to fill data across. Review dim_partial_title_matches_review.xlsx manually
# and add any Dim index values that are legitimately new data to the exclusion set.

############################################################################################
# Add Dim index values here to skip — they will NOT be removed from dimensions_data.
excluded_dim_partial_indices = set()
excluded_dim_partial_indices = {0, 5, 8, 25, 37, 38}
############################################################################################

dim_partial_matched_indices = set()
for _, match in dim_partial_matches_df.iterrows():
    dim_idx = int(match['Dim index'])
    if dim_idx not in excluded_dim_partial_indices:
        dim_partial_matched_indices.add(dim_idx)

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data.index.isin(dim_partial_matched_indices)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Processed {len(dim_partial_matched_indices)} partial matches ({len(excluded_dim_partial_indices)} excluded)")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Processed 0 partial matches (6 excluded)
Removed 0 rows from dimensions_data (173 remaining)


In [30]:
from pathlib import Path

output_dir = Path('1_deduplication/data_output')
output_dir.mkdir(parents=True, exist_ok=True)

last_year_data_edited.to_excel(output_dir / 'last_year_data_edited.xlsx', index=False)
print(f'Saved last_year_data_edited ({len(last_year_data_edited)} rows) -> {output_dir / "last_year_data_edited.xlsx"}')

dimensions_data.to_excel(output_dir / 'dimensions_data_filtered.xlsx', index=False)
print(f'Saved dimensions_data ({len(dimensions_data)} rows) -> {output_dir / "dimensions_data_filtered.xlsx"}')

Saved last_year_data_edited (1509 rows) -> 1_deduplication\data_output\last_year_data_edited.xlsx
Saved dimensions_data (173 rows) -> 1_deduplication\data_output\dimensions_data_filtered.xlsx
